# 01. Data Exploration and Visualization — Inter Dataset Analysis

This notebook performs a **full inter-dataset comparison of Normal and Adjacent** breast tissues across three GEO DNA-methylation cohorts (GSE69914, GSE225845, GSE287331). After loading and aligning β-matrices with their matched phenotype tables, it restricts all analyses to shared CpG sites and harmonized sample groups. The workflow computes Δβ(Adjacent–Normal) profiles for each dataset, evaluates their cross-cohort concordance through kernel densities and Spearman correlations, and visualizes the joint structure via PCA and t-SNE embeddings. 

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     01-data-exploration-visualization-inter-analysis   ║
# ║ Description:  Inter-dataset comparison of Normal vs Adjacent     ║
# ║               breast tissues across three GEO methylation        ║
# ║               cohorts, focusing on shared-CpG structure,         ║
# ║               Δβ concordance, and PCA/t-SNE embeddings.          ║
# ║ Dataset(s):   GSE69914, GSE225845, GSE287331                     ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 19-Nov-2025 | Language: Python 3.11.13                     ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [1]:
!sudo apt-get update -qq
!sudo apt-get install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  cm-super-minimal dvisvgm fonts-lato fonts-lmodern fonts-texgyre
  libapache-pom-java libcommons-logging-java libcommons-parent-java
  libfontbox-java libkpathsea6 libpdfbox-java libptexenc1 libruby3.0
  libsynctex2 libteckit0 libtexlua53 libtexluajit2 libwoff1 libzzip-0-13
  lmodern pfb2t1c2pfb preview-latex-style rake ruby ruby-net-telnet
  ruby-rubygems ruby-webrick ruby-xmlrpc ruby3.0 rubygems-integration t1utils
  tex-common tex-gyre texlive-base texlive-binaries texlive-latex-base
  texlive-latex-recommended texlive-pictures texlive-plain-generic tipa
  xfonts-encodings xfonts-utils
Suggested packages:
  libavalon-framework-java libcommons-loggin

In [5]:
from __future__ import annotations
import json, time, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import polars as pl
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.manifold import TSNE


## Inter Dataset Comparison

In [8]:
# INTER-DATASET COMPARISON: NORMAL VS ADJACENT ONLY
# Setting 
DATASET_1 = "GSE69914"
DATASET_2 = "GSE287331"
DATASET_3 = "GSE225845"

PATH_BETA_1  = Path("/kaggle/input/gse69914-parquet/GSE69914.parquet")
PATH_PHENO_1 = Path("/kaggle/input/pheno-gse69914/pheno_GSE69914.parquet")

PATH_BETA_2  = Path("/kaggle/input/gse225845-parquet/GSE225845.parquet")
PATH_PHENO_2 = Path("/kaggle/input/pheno-gse225845/pheno_GSE225845.parquet")

PATH_BETA_3  = Path("/kaggle/input/3-gse287331-parquet/GSE287331_clean_imputed.parquet")
PATH_PHENO_3 = Path("/kaggle/input/1-pheno-gse287331-parquet/pheno_GSE287331.parquet")

BETA_ID_COL    = "id_tissue"
BETA_LABEL_COL = "label"      # optional in β; main label comes from pheno

TRIPLE_NAME = f"{DATASET_1}_{DATASET_2}_{DATASET_3}"
OUTPUT_DIR  = Path(f"./INTERDATASET_{TRIPLE_NAME}")

USE_TEX = True
SEED    = 42

LABEL_MAP   = {0: "Normal", 1: "Adjacent"}
GROUP_ORDER = ["Normal", "Adjacent"]

PCA_TOP_CPGS     = 20000   # variance-ranked CpGs from intersection
IPCA_COMPONENTS  = 80      # for t-SNE pre-reduction
TSNE_PERPLEXITY  = 30
TSNE_N_ITER      = 1500

# Utils
def info(msg: str): print(f"[INFO] {msg}")
def ok(msg: str):   print(f"[OK]   {msg}")
def warn(msg: str): print(f"[WARN] {msg}")

def seed_everything(seed: int = SEED):
    np.random.seed(seed)

def ensure_dirs():
    for sub in ["figures", "tables", "logs"]:
        (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

def _has_latex() -> bool:
    if not USE_TEX:
        return False
    return bool(shutil.which("pdflatex") or shutil.which("xelatex") or shutil.which("lualatex"))

def L(text: str | None) -> str | None:
    if text is None:
        return None
    t = str(text)
    if not _has_latex():
        return (
            t.replace("β", "beta")
             .replace("Δβ", "Delta beta")
             .replace("Δ", "Delta")
             .replace("μ", "mu")
             .replace("×", "x")
        )
    t = (
        t.replace("Δβ", r"$\Delta\beta$")
         .replace("β", r"$\beta$")
         .replace("Δ", r"$\Delta$")
         .replace("μ", r"$\mu$")
         .replace("×", r"$\times$")
    )
    specials = {
        "&": r"\&", "%": r"\%", "_": r"\_", "#": r"\#",
        "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}"
    }
    out = []
    in_math = False
    for ch in t:
        if ch == "$":
            in_math = not in_math
            out.append(ch)
            continue
        if in_math or ch not in specials:
            out.append(ch)
        else:
            out.append(specials[ch])
    return "".join(out)

def apply_thesis_style(use_tex: bool = True, legend_position: str = "upper right"):
    sns.set_theme(style="whitegrid", context="notebook")
    tex_flag = bool(use_tex and _has_latex())
    mpl.rcParams.update({
        "text.usetex": tex_flag,
        "font.family": "serif",
        "font.serif": ["Computer Modern Roman", "Latin Modern Roman", "Times New Roman"],
        "mathtext.fontset": "cm",
        "figure.figsize": (6.8, 4.5),
        "font.size": 8.5,
        "axes.labelsize": 10.5,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9.5,
        "legend.title_fontsize": 9.5,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.facecolor": "white",
        "axes.edgecolor": "#D0D0D0",
        "axes.linewidth": 0.8,
        "legend.frameon": True,
        "legend.facecolor": "white",
        "legend.edgecolor": "#D0D0D0",
        "legend.loc": legend_position,
        "legend.framealpha": 1.0,
    })
    def place_legend(ax=None, position: str = "upper right"):
        if ax is None:
            ax = plt.gca()
        leg = ax.legend(
            loc=position,
            frameon=True,
            facecolor="white",
            edgecolor=mpl.rcParams.get("legend.edgecolor", "#D0D0D0"),
            framealpha=1.0,
            handlelength=1.8,
            handletextpad=0.6,
            borderpad=0.4,
            fancybox=False,
        )
        if leg:
            leg.get_frame().set_linewidth(0.8)
        return ax
    globals()["place_legend"] = place_legend  # inject globally

PLOT_ID = 1
def savefig(name: str):
    global PLOT_ID
    fig = plt.gcf()
    for ax in fig.axes:
        for sp in ax.spines.values():
            sp.set_visible(True)
            sp.set_linewidth(0.8)
            sp.set_edgecolor(mpl.rcParams.get("axes.edgecolor", "#D0D0D0"))
    out = OUTPUT_DIR / "figures" / f"{TRIPLE_NAME}_{PLOT_ID:02d}_{name}.pdf"
    plt.tight_layout()
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    ok(f"Saved figure → {out}")
    PLOT_ID += 1


# Data loading
def load_beta_parquet(path: Path) -> pd.DataFrame:
    t0 = time.time()
    lf = pl.scan_parquet(str(path))
    df_pl = lf.collect()
    if BETA_ID_COL not in df_pl.columns:
        raise KeyError(f"[{path.name}] '{BETA_ID_COL}' not found. Available: {df_pl.columns}")
    df_pl = df_pl.rename({BETA_ID_COL: "id_tissue"})
    if BETA_LABEL_COL in df_pl.columns:
        df_pl = df_pl.with_columns(pl.col(BETA_LABEL_COL).cast(pl.Int64))
    exclude_cols = ["id_tissue"]
    if BETA_LABEL_COL in df_pl.columns:
        exclude_cols.append(BETA_LABEL_COL)
    df_pl = df_pl.with_columns(pl.all().exclude(exclude_cols).cast(pl.Float32))
    df = df_pl.to_pandas()
    beta = df.set_index("id_tissue").drop(
        columns=[c for c in ["label", BETA_LABEL_COL] if c in df.columns],
        errors="ignore",
    )
    ok(f"β loaded from {path.name} → shape={beta.shape} in {time.time()-t0:0.2f}s")
    return beta

def load_pheno(path: Path) -> pd.DataFrame:
    t0 = time.time()
    if path.suffix.lower() == ".parquet":
        ph = pl.scan_parquet(str(path)).collect().to_pandas()
    else:
        ph = pd.read_csv(path)
    ph.columns = [c.strip() for c in ph.columns]
    if not {"id_tissue", "label"}.issubset(ph.columns):
        raise KeyError(f"[{path.name}] pheno must have id_tissue,label; got {ph.columns}")
    ok(f"pheno loaded from {path.name} (rows={len(ph)}, cols={len(ph.columns)}) in {time.time()-t0:0.2f}s")
    return ph

def align_beta_pheno_normal_adj(beta: pd.DataFrame, ph: pd.DataFrame):
    """
    Keep only Normal (0) and Adjacent (1) samples.
    Return: beta_filtered, labels_string, pheno_filtered
    """
    ph2 = ph[ph["label"].isin([0, 1])].copy()
    common = beta.index.intersection(ph2["id_tissue"])
    ph2 = ph2.set_index("id_tissue").loc[common].copy()
    beta2 = beta.loc[common].copy()
    labels = pd.to_numeric(ph2["label"], errors="coerce").astype("Int64").map(LABEL_MAP)
    return beta2, labels, ph2

# Intersection & Δβ
def intersect_cpgs(b1: pd.DataFrame, b2: pd.DataFrame, b3: pd.DataFrame) -> list[str]:
    inter = list(set(b1.columns).intersection(b2.columns).intersection(b3.columns))
    inter.sort()
    return inter

def to_M_values(beta: pd.DataFrame) -> pd.DataFrame:
    B = beta.clip(lower=1e-6, upper=1-1e-6)
    return np.log2(B / (1 - B))

def delta_beta_vector(beta: pd.DataFrame, labels: pd.Series,
                      g1: str = "Adjacent", g2: str = "Normal") -> pd.Series | None:
    idx1 = labels[labels == g1].index
    idx2 = labels[labels == g2].index
    if len(idx1) == 0 or len(idx2) == 0:
        warn(f"No samples for {g1} or {g2}; skipping Δβ.")
        return None
    return beta.loc[idx1].mean(axis=0) - beta.loc[idx2].mean(axis=0)

def delta_beta_overlay_three(d1, d2, d3,
                             name1=DATASET_1,
                             name2=DATASET_2,
                             name3=DATASET_3,
                             g1="Adjacent", g2="Normal"):
    plt.figure(figsize=(6.8, 4.5))
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, 3))

    for delta, name, col in [(d1, name1, colors[0]),
                             (d2, name2, colors[1]),
                             (d3, name3, colors[2])]:
        if delta is None:
            continue
        x = delta.dropna().to_numpy(np.float32)
        sns.kdeplot(
            x, bw_adjust=0.9, linewidth=1.6,
            alpha=0.9, color=col,
            label=L(f"{name}")
        )

    plt.axvline(0, ls="--", lw=1.0, color="black", label=L("Zero reference"))
    plt.xlim(-0.5, 0.5)
    plt.xlabel(L(f"$\\Delta\\beta$ ({g1} - {g2})"))
    plt.ylabel(L("Density"))
    place_legend(position="upper right")
    savefig("delta_beta_AminusN_overlay_three")


def pairwise_delta_scatter(dA: pd.Series, dB: pd.Series,
                           nameA: str, nameB: str,
                           g1: str = "Adjacent", g2: str = "Normal"):
    if dA is None or dB is None:
        return None
    common = dA.dropna().index.intersection(dB.dropna().index)
    if len(common) == 0:
        warn(f"No common CpGs for {nameA} vs {nameB} Δβ scatter.")
        return None
    x = dA.loc[common].to_numpy(np.float32)
    y = dB.loc[common].to_numpy(np.float32)
    rho, p = stats.spearmanr(x, y)

    plt.figure(figsize=(6.6, 6.2))
    plt.scatter(
        x, y, s=5, alpha=0.3,
        c=plt.cm.viridis(0.55),
        edgecolors="none",
        label=L("CpGs (intersection)")
    )
    lim = 1.0
    plt.plot([-lim, lim], [-lim, lim], ls="--", lw=1.0, color="black", label=L("y = x"))
    plt.xlim(-lim, lim)
    plt.ylim(-lim, lim)
    plt.xlabel(L(f"$\\Delta\\beta$ {nameA} ({g1} - {g2})"))
    plt.ylabel(L(f"$\\Delta\\beta$ {nameB} ({g1} - {g2})"))
    plt.text(
        0.02, 0.95, L(f"Spearman $\\rho$ = {rho:.3f}"),
        transform=plt.gca().transAxes,
        ha="left", va="top"
    )
    place_legend()
    savefig(f"delta_beta_scatter_{nameA}_vs_{nameB}_{g1}_minus_{g2}")
    return float(rho), int(len(common))


# Embeddings on intersection (N + A)
def pca_tsne_on_intersection(bI: pd.DataFrame,
                             labels_concat: pd.Series,
                             dataset_ids: pd.Series):
    var = bI.var(axis=0, ddof=1)
    keep = var.nlargest(min(PCA_TOP_CPGS, var.size)).index
    X = to_M_values(bI.loc[:, keep])
    X = X - X.mean(axis=0)

    pca = PCA(n_components=2, random_state=SEED)
    Xp = pca.fit_transform(X)
    dfp = pd.DataFrame(Xp, columns=["PC1", "PC2"], index=bI.index)
    dfp["group"] = labels_concat
    dfp["dataset"] = dataset_ids

    ds_colors = {
        DATASET_1: plt.cm.viridis(0.20),
        DATASET_2: plt.cm.viridis(0.50),
        DATASET_3: plt.cm.viridis(0.80),
    }
    gp_colors = {g: c for g, c in zip(GROUP_ORDER, plt.cm.viridis(np.linspace(0.15, 0.85, 2)))}

    # PCA by dataset
    plt.figure(figsize=(6.2, 5.4))
    for ds, col in ds_colors.items():
        dsub = dfp[dfp["dataset"] == ds]
        plt.scatter(dsub["PC1"], dsub["PC2"], s=18, alpha=0.8, color=col, label=L(ds))
    plt.xlabel(L("Principal Component 1"))
    plt.ylabel(L("Principal Component 2"))
    place_legend()
    savefig("pca_by_dataset_NA")

    # PCA by group (Normal vs Adjacent)
    plt.figure(figsize=(6.2, 5.4))
    for g, col in gp_colors.items():
        dsub = dfp[dfp["group"] == g]
        if len(dsub) == 0:
            continue
        plt.scatter(dsub["PC1"], dsub["PC2"], s=18, alpha=0.8, color=col, label=L(f"{g} samples"))
    plt.xlabel(L("Principal Component 1"))
    plt.ylabel(L("Principal Component 2"))
    place_legend()
    savefig("pca_by_group_NA")

    # t-SNE
    ipca = IncrementalPCA(n_components=IPCA_COMPONENTS, batch_size=5000)
    Xred = ipca.fit_transform(X)
    tsne = TSNE(
        n_components=2,
        init="pca",
        perplexity=TSNE_PERPLEXITY,
        learning_rate=300,
        early_exaggeration=12,
        n_iter=TSNE_N_ITER,
        metric="euclidean",
        random_state=SEED,
    )
    Xt = tsne.fit_transform(Xred)
    dft = pd.DataFrame(Xt, columns=["tSNE1", "tSNE2"], index=bI.index)
    dft["group"] = labels_concat
    dft["dataset"] = dataset_ids

    # t-SNE by dataset
    plt.figure(figsize=(6.2, 5.4))
    for ds, col in ds_colors.items():
        dsub = dft[dft["dataset"] == ds]
        plt.scatter(dsub["tSNE1"], dsub["tSNE2"], s=18, alpha=0.8, color=col, label=L(ds))
    plt.xlabel(L("t-Distributed Stochastic Neighbor Embedding 1"))
    plt.ylabel(L("t-Distributed Stochastic Neighbor Embedding 2"))
    place_legend(position="upper left")
    savefig("tsne_by_dataset_NA")

    # t-SNE by group
    plt.figure(figsize=(6.2, 5.4))
    for g, col in gp_colors.items():
        dsub = dft[dft["group"] == g]
        if len(dsub) == 0:
            continue
        plt.scatter(dsub["tSNE1"], dsub["tSNE2"], s=18, alpha=0.8, color=col, label=L(f"{g} samples"))
    plt.xlabel(L("t-Distributed Stochastic Neighbor Embedding 1"))
    plt.ylabel(L("t-Distributed Stochastic Neighbor Embedding 2"))
    place_legend(position="upper left")
    savefig("tsne_by_group_NA")

# Summary
def write_summary(path: Path, meta: dict, overlap: dict, delta_corrs: dict):
    lines = []
    A = meta["A"]; B = meta["B"]; C = meta["C"]
    lines.append(f"INTER-DATASET SUMMARY — {TRIPLE_NAME}")
    lines.append("=" * 72)
    lines.append("")
    lines.append(f"Datasets: {DATASET_1}, {DATASET_2}, {DATASET_3}")
    lines.append("")
    lines.append("Sample counts by dataset (Normal / Adjacent):")
    lines.append(f"- {DATASET_1}: {A['n_samples']} total | N={A['n_N']} | Adj={A['n_A']}")
    lines.append(f"- {DATASET_2}: {B['n_samples']} total | N={B['n_N']} | Adj={B['n_A']}")
    lines.append(f"- {DATASET_3}: {C['n_samples']} total | N={C['n_N']} | Adj={C['n_A']}")
    lines.append("")
    lines.append("CpG sets:")
    lines.append(f"- |CpGs| {DATASET_1}: {A['n_cpg']:,}")
    lines.append(f"- |CpGs| {DATASET_2}: {B['n_cpg']:,}")
    lines.append(f"- |CpGs| {DATASET_3}: {C['n_cpg']:,}")
    lines.append(f"- Intersection (all three): {overlap['n_intersection']:,}")
    lines.append("")
    lines.append("Δβ (Adjacent–Normal) correlations (Spearman ρ) on intersection:")
    for k, v in delta_corrs.items():
        lines.append(f"- {k}: ρ = {v['rho']:.3f} (n={v['n']})")
    lines.append("")
    lines.append("All figures saved under 'figures/' with thesis style.")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    ok(f"summary.txt written → {path}")


In [9]:
# MAIN
warnings.filterwarnings("ignore")
seed_everything(SEED)
ensure_dirs()
apply_thesis_style(USE_TEX)

# Load and align (Normal + Adjacent only) 
info(f"Load β: {DATASET_1}")
b1_full = load_beta_parquet(PATH_BETA_1)
info(f"Load pheno: {DATASET_1}")
p1_full = load_pheno(PATH_PHENO_1)
b1, l1, p1 = align_beta_pheno_normal_adj(b1_full, p1_full)
del b1_full, p1_full

info(f"Load β: {DATASET_2}")
b2_full = load_beta_parquet(PATH_BETA_2)
info(f"Load pheno: {DATASET_2}")
p2_full = load_pheno(PATH_PHENO_2)
b2, l2, p2 = align_beta_pheno_normal_adj(b2_full, p2_full)
del b2_full, p2_full

info(f"Load β: {DATASET_3}")
b3_full = load_beta_parquet(PATH_BETA_3)
info(f"Load pheno: {DATASET_3}")
p3_full = load_pheno(PATH_PHENO_3)
b3, l3, p3 = align_beta_pheno_normal_adj(b3_full, p3_full)
del b3_full, p3_full

ok(f"{DATASET_1}: samples={b1.shape[0]}, CpGs={b1.shape[1]:,}")
ok(f"{DATASET_2}: samples={b2.shape[0]}, CpGs={b2.shape[1]:,}")
ok(f"{DATASET_3}: samples={b3.shape[0]}, CpGs={b3.shape[1]:,}")

# CpG intersection 
inter = intersect_cpgs(b1, b2, b3)
ok(f"CpG intersection (3-way): {len(inter):,}")
pd.Series(inter, name="CpG").to_csv(
    OUTPUT_DIR / "tables" / "cpg_intersection_3way_NA.csv", index=False
)

counts_tbl = pd.DataFrame({
    "dataset": [DATASET_1, DATASET_2, DATASET_3, "Intersection(3)"],
    "n_cpg":  [b1.shape[1], b2.shape[1], b3.shape[1], len(inter)],
})
counts_tbl.to_csv(OUTPUT_DIR / "tables" / "cpg_counts_NA.csv", index=False)

# Δβ (Adjacent–Normal) per dataset 
info("Δβ (Adjacent–Normal) per dataset + pairwise comparisons…")
d1 = delta_beta_vector(b1, l1, "Adjacent", "Normal")
d2 = delta_beta_vector(b2, l2, "Adjacent", "Normal")
d3 = delta_beta_vector(b3, l3, "Adjacent", "Normal")

delta_beta_overlay_three(d1, d2, d3,
                         name1=DATASET_1,
                         name2=DATASET_2,
                         name3=DATASET_3,
                         g1="Adjacent", g2="Normal")

# Save Δβ vectors
if d1 is not None:
    d1.to_frame("delta_beta_AminusN").to_csv(
        OUTPUT_DIR / "tables" / f"delta_beta_{DATASET_1}_AminusN.csv"
    )
if d2 is not None:
    d2.to_frame("delta_beta_AminusN").to_csv(
        OUTPUT_DIR / "tables" / f"delta_beta_{DATASET_2}_AminusN.csv"
    )
if d3 is not None:
    d3.to_frame("delta_beta_AminusN").to_csv(
        OUTPUT_DIR / "tables" / f"delta_beta_{DATASET_3}_AminusN.csv"
    )

# Pairwise Δβ scatter on intersection
delta_corrs = {}

if d1 is not None and d2 is not None:
    out = pairwise_delta_scatter(d1, d2, DATASET_1, DATASET_2,
                                 g1="Adjacent", g2="Normal")
    if out is not None:
        rho12, n12 = out
        delta_corrs[f"{DATASET_1} vs {DATASET_2}"] = {"rho": rho12, "n": n12}

if d1 is not None and d3 is not None:
    out = pairwise_delta_scatter(d1, d3, DATASET_1, DATASET_3,
                                 g1="Adjacent", g2="Normal")
    if out is not None:
        rho13, n13 = out
        delta_corrs[f"{DATASET_1} vs {DATASET_3}"] = {"rho": rho13, "n": n13}

if d2 is not None and d3 is not None:
    out = pairwise_delta_scatter(d2, d3, DATASET_2, DATASET_3,
                                 g1="Adjacent", g2="Normal")
    if out is not None:
        rho23, n23 = out
        delta_corrs[f"{DATASET_2} vs {DATASET_3}"] = {"rho": rho23, "n": n23}

# Embeddings on intersection (Normal + Adjacent only)
info("Embeddings on intersection (Normal + Adjacent)…")
B1I = b1.loc[:, inter]
B2I = b2.loc[:, inter]
B3I = b3.loc[:, inter]
BI  = pd.concat([B1I, B2I, B3I], axis=0, copy=False)

labels_concat = pd.concat([l1, l2, l3], axis=0)
dataset_ids = pd.Series(
    [DATASET_1] * B1I.shape[0] +
    [DATASET_2] * B2I.shape[0] +
    [DATASET_3] * B3I.shape[0],
    index=BI.index,
    name="dataset"
)

pca_tsne_on_intersection(BI, labels_concat, dataset_ids)

# Summary
meta = {
    "A": {
        "n_samples": int(b1.shape[0]),
        "n_cpg": int(b1.shape[1]),
        "n_N": int((l1 == "Normal").sum()),
        "n_A": int((l1 == "Adjacent").sum()),
    },
    "B": {
        "n_samples": int(b2.shape[0]),
        "n_cpg": int(b2.shape[1]),
        "n_N": int((l2 == "Normal").sum()),
        "n_A": int((l2 == "Adjacent").sum()),
    },
    "C": {
        "n_samples": int(b3.shape[0]),
        "n_cpg": int(b3.shape[1]),
        "n_N": int((l3 == "Normal").sum()),
        "n_A": int((l3 == "Adjacent").sum()),
    },
}
overlap = {"n_intersection": int(len(inter))}

write_summary(OUTPUT_DIR / "logs" / "summary_NA.txt", meta, overlap, delta_corrs)

ok("Done. All artifacts saved under: " + str(OUTPUT_DIR.resolve()))


[INFO] Load β: GSE69914
[OK]   β loaded from GSE69914.parquet → shape=(407, 485512) in 25.39s
[INFO] Load pheno: GSE69914
[OK]   pheno loaded from pheno_GSE69914.parquet (rows=407, cols=10) in 0.01s
[INFO] Load β: GSE287331
[OK]   β loaded from GSE225845.parquet → shape=(477, 750426) in 34.22s
[INFO] Load pheno: GSE287331
[OK]   pheno loaded from pheno_GSE225845.parquet (rows=477, cols=10) in 0.01s
[INFO] Load β: GSE225845
[OK]   β loaded from GSE287331_clean_imputed.parquet → shape=(446, 703166) in 30.67s
[INFO] Load pheno: GSE225845
[OK]   pheno loaded from pheno_GSE287331.parquet (rows=446, cols=5) in 0.01s
[OK]   GSE69914: samples=92, CpGs=485,512
[OK]   GSE287331: samples=253, CpGs=750,426
[OK]   GSE225845: samples=242, CpGs=703,166
[OK]   CpG intersection (3-way): 326,330
[INFO] Δβ (Adjacent–Normal) per dataset + pairwise comparisons…
[OK]   Saved figure → INTERDATASET_GSE69914_GSE287331_GSE225845/figures/GSE69914_GSE287331_GSE225845_01_delta_beta_AminusN_overlay_three.pdf
[OK]  